# ex-2d: Heavy-tailed sin²∠ histograms across growing p

This notebook shows the empirical distribution of the finite-p sine-squared angle quantity, 'sin2_j', across simulation reps as p grows, separated by factor. It is a standalone histogram view of the same observable-floor setup used in ex-2c / ex-3, so the model, design choices, and random seed are reused directly for comparability. No '.py' files were changed for this notebook; all work here is notebook-only.

## Growing p (fixed n = 63), heavy-tailed returns 
The sweep generates 'sin^2_j' values across reps for a fixed sample size 'n = 63' under the heavy-tailed Student-t return design, then groups them by p and factor 'j = 1, 2, 3'. We use 'p ∈ {100, 500, 5000}' to show the small / intermediate / larger-p regime in a compact 3×3 layout without changing the underlying simulation setup.

## Change log 
- New standalone notebook; nothing in ex-2c/ex-3 or any '.py' file modified
- Reuses the canonical model + '\*\*HEAVY_TAIL' (Student-t) + 'SEED' from ex-2c/ex-3 
- 9-panel 'sin²∠' histograms, 'p ∈ {100, 500, 5000}' × factors 1–3, 'R = 1000' 
- Under the heavy-tailed design, there is still a visible right-edge 'sin²∠ ≈ 1' pile-up for part of the grid, so the angle distribution does not fully wash out even as p increases

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd()
OUT_DIR = REPO_ROOT / "nb_outputs"
OUT_DIR.mkdir(exist_ok=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from loguru import logger; logger.remove()   # quiet notebook
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from fl_experiment_setup import ModelSpec, DesignSpec
from fl_experiment_runner import run_experiment
from sim_corollary_obs_floor import ObservableFloorExperiment

# canonical model — same calibration as ex-2c / ex-3
model = ModelSpec(
    k_factors=3,
    factor_vols=[0.16, 0.08, 0.06],
    beta_samplers=[
        {"distribution": "normal", "loc": 1.0, "scale": 0.5},   # market-like factor 1
        {"distribution": "normal", "loc": 0.0, "scale": 1.0},
        {"distribution": "normal", "loc": 0.0, "scale": 1.0},
    ],
    idio_vol_sampler={"distribution": "constant", "value": 0.4},
)

HEAVY_TAIL = dict(
    factor_return_sampler={"distribution": "student_t", "df": 6, "loc": 0.0, "scale": 1.0},
    idio_return_sampler={"distribution": "student_t", "df": 5, "loc": 0.0, "scale": 1.0},
)

N_REPS, SEED = 1000, 20260511

_ret_lab = rf"$\mathcal{{T}}_{{{HEAVY_TAIL['factor_return_sampler']['df']}}}$"
_idio_lab = rf"$\mathcal{{T}}_{{{HEAVY_TAIL['idio_return_sampler']['df']}}}$"
print("ready")

In [ ]:
design_p_standalone = DesignSpec(
    n_values=[63],              # one time-length, 63 days
    p_values=[100, 500, 5000],        
    n_reps=N_REPS, random_seed=SEED, sampling="nested", nest_time=True, **HEAVY_TAIL,
)

df_p_standalone = run_experiment(model, design_p_standalone, ObservableFloorExperiment())
print(df_p_standalone.shape)
print(df_p_standalone.columns.tolist())
df_p_standalone.head()

In [ ]:
expected = N_REPS * len(design_p_standalone.n_values) * len(design_p_standalone.p_values) * model.k_factors
print("rows:", len(df_p_standalone), "expected:", expected, "match:", len(df_p_standalone) == expected)

print("sin2 in [0,1]:", df_p_standalone["sin2_j"].between(0, 1).all())
print("dup (n,p,j) counts unique?:", (df_p_standalone.groupby(["n","p","j"]).size().nunique() == 1))

In [ ]:
p_list = [100, 500, 5000]      # the three rows
j_list = [1, 2, 3]       # the three columns

fig, axes = plt.subplots(3, 3, figsize=(12, 9))   # a 3×3 grid of empty boxes

for row, p in enumerate(p_list):       
    for col, j in enumerate(j_list):   
        ax = axes[row, col]             
        data = df_p_standalone[(df_p_standalone["p"] == p) & (df_p_standalone["j"] == j)]["sin2_j"]   # sin² for this p and j
        ax.hist(data, bins=30)          # draw the histogram into that box
        ax.set_title(f"factor {j},  p={p:,}")

fig.suptitle(
    rf"Growing p, fixed n=63 — distribution of sin²∠(h, b̄) across 1000 reps  (Factor Returns ~ {_ret_lab}, Idiosyncratic ~ {_idio_lab})",
    y=0.98,
)
fig.supxlabel("sin²∠(h, b̄)")
fig.tight_layout(rect=[0, 0, 1, 0.96])   # rect leaves room so the suptitle doesn't overlap row 1

fig.savefig(OUT_DIR / "ex2d_sin2_histograms.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "ex2d_sin2_histograms.pdf", bbox_inches="tight")
plt.show()